# Growth on unserved demand, quick first cut

**What changed from v3:**
- **Growth areas cover every named OSM place,** including ones our lounges already reach. Before, only unreached cells counted.
- **GROW** = at least `UNSERVED_MIN` addressable women that no premium supplier serves (Huff, `served_demand.ipynb`), with worker housing under 50% and a residential name.
- **The premium-saturation test is gone.** Competitors already take their Huff share before anything counts as unserved.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_rows", 80)
from src.baseline import run
from src.data_v3 import load_v3
from src.config import REPO_ROOT, load_baseline_assumptions
from src.model import served as S, scorecard, growth
r = run(); v3 = load_v3(); a = load_baseline_assumptions(REPO_ROOT / "data/scenarios/baseline.yaml")
old = pd.read_csv(REPO_ROOT / "data/scenarios/baseline_v3_labels.csv")
old_action = dict(zip(old.id, old.action))

## 1. GROW and WATCH

In [2]:
A = {x.area_id: x for x in r.areas}
t = pd.DataFrame([dict(area=d.area_id, emirate=A[d.area_id].emirate, call=d.action, v3=old_action.get(d.area_id, "(new area)"),
                       women=round(A[d.area_id].addressable_women), unserved=round(A[d.area_id].unserved_women),
                       unserved_share=round(A[d.area_id].unserved_women / max(A[d.area_id].addressable_women, 1), 2),
                       nearest_km=round(A[d.area_id].nearest_lounge_km, 1), prem_per_1k=A[d.area_id].premium_reviews_per_1k,
                       confidence=d.confidence) for d in r.area_decisions if d.action != "SKIP"])
print("UNSERVED_MIN:", growth.UNSERVED_MIN, "|", pd.Series([d.action for d in r.area_decisions]).value_counts().to_dict())
t.sort_values(["call", "unserved"], ascending=[True, False]).set_index("area")

UNSERVED_MIN: 10000 | {'SKIP': 535, 'WATCH': 15, 'GROW': 11}


,emirate,call,v3,women,unserved,unserved_share,nearest_km,prem_per_1k,confidence
area,,,,,,,,,
al-dhaid-sharjah,Sharjah,GROW,GROW,73067,50570,0.69,43.8,5.433826,high
al-madam-sharjah,Sharjah,GROW,WATCH,38980,38980,1.00,46.7,1.050737,high
al-batayih-sharjah,Sharjah,GROW,WATCH,21235,17396,0.82,28.8,5.580992,high
al-lesaily-dubai,Dubai,GROW,WATCH,17231,16795,0.97,27.1,2.936039,high
al-hiyar-abu-dhabi,Abu Dhabi,GROW,SKIP,15385,15385,1.00,38.9,NaN,high
milehah-sharjah,Sharjah,GROW,WATCH,13716,13369,0.97,43.3,0.000000,high
sharjah-sharjah,Sharjah,GROW,GROW,281946,13220,0.05,2.5,116.415831,high
mzeer-ah-abu-dhabi,Abu Dhabi,GROW,SKIP,12678,12678,1.00,55.0,NaN,high
rimah-abu-dhabi,Abu Dhabi,GROW,SKIP,12520,12429,0.99,33.0,NaN,high


## 2. Inside our footprint or remote?

In [3]:
g = t[t.call == "GROW"]
print("GROW within 15 km of a lounge:", list(g.area[g.nearest_km <= 15]))
print("GROW 30+ km from any lounge:", list(g.area[g.nearest_km >= 30]))
print("median unserved share, GROW areas:", g.unserved_share.median())

GROW within 15 km of a lounge: ['sharjah-sharjah']
GROW 30+ km from any lounge: ['al-dhaid-sharjah', 'al-madam-sharjah', 'zayed-city-abu-dhabi', 'al-hiyar-abu-dhabi', 'milehah-sharjah', 'mzeer-ah-abu-dhabi', 'rimah-abu-dhabi', 'ghiyathi-abu-dhabi']
median unserved share, GROW areas: 0.97


## 3. v3 → new, for the v3 GROW and WATCH areas

In [4]:
v3_areas = old[(old.kind == "area") & old.action.isin(["GROW", "WATCH"])]
new_action = {d.area_id: d.action for d in r.area_decisions}
pd.DataFrame({"v3": v3_areas.set_index("id").action, "new": pd.Series({i: new_action.get(i, "(merged/absent)") for i in v3_areas.id})})

,v3,new
sharjah-sharjah,GROW,GROW
al-dhaid-sharjah,GROW,GROW
al-madam-sharjah,WATCH,GROW
khor-fakkan-sharjah,GROW,WATCH
kalba-sharjah,GROW,WATCH
al-jerf-ajman,GROW,SKIP
al-batayih-sharjah,WATCH,GROW
al-awir-dubai,WATCH,SKIP
al-lesaily-dubai,WATCH,GROW
hamidiya-ajman,WATCH,SKIP


## 4. Open questions for review

1. **Remote, empty places still win.** 8 of the 11 GROW areas are 30+ km from any lounge, and most are 97–100% unserved, because cells no supplier reaches count as fully unserved. That's how Ghiyathi (99 km) and Mzeer'ah (55 km) become GROW. Is that a market, or just no salons because there's no market? Options:
   - only count unserved demand within some distance of existing retail;
   - require a minimum density;
   - shrink the outside option for unreached cells.
2. **Sharjah city is GROW from inside our footprint** (2.5 km from a lounge). That's the change the served-demand model was meant to allow.
3. **`UNSERVED_MIN` = 10k** is "about what a mid-sized lounge serves". It's a first cut that needs agreeing.
4. **Area confidence** uses distance to the line only. Assumption flips (the kalba finding) are not in yet.